# Task 3: Jointly Trained Soft Mixture-of-Experts (SoftMoERestorationNetwork)

**Generative AI Restoration Studio — Milestone 3b**

This notebook implements:
1. **SoftMoERestorationNetwork**: 4 expert branches (Clean identity bypass + 3 specialist autoencoders) controlled by a temperature-scaled softmax gate network.
2. **Warm-Start Initialization**: Pretrained weights from Task 2 classifier and specialists.
3. **Two-Phase Training**: Phase 1 (Gate warm-up with frozen experts) -> Phase 2 (Joint end-to-end fine-tuning).
4. **4-Term Composite Loss**: $\mathcal{L}_{\text{total}} = \lambda_1 \mathcal{L}_{\text{recon}} + \lambda_2 \mathcal{L}_{\text{class}} + \lambda_3 \mathcal{L}_{\text{balance}} + \lambda_4 \mathcal{L}_{\text{entropy}}$.
5. **Optuna 15-Trial Study**: Hyperparameter tuning for $\tau$, $\eta_{\text{joint}}$, and composite loss weights.
6. **4-Way Benchmark**: Comprehensive evaluation against Universal (Task 1), Oracle Hard, and Predicted Hard (Task 2) across 3,669 test images.
7. **Gating Analysis**: Weight distribution heatmaps and routing behavior analysis across corruptions.
8. **Self-Contained ONNX Export**: Opset 18 multi-output export with numerical parity verification.

In [ ]:
# Step 1: Install Dependencies & Verify GPU
!pip install -q onnx onnxruntime onnxscript mlflow optuna matplotlib seaborn
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 2: Google Drive Mounting & Fast Local Disk Caching
import os, sys, shutil, time
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/GenAI-A1"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"Current working directory: {os.getcwd()}")

LOCAL_DATA_DIR = "/content/local_data/OxfordPet/images_128x128"
DRIVE_DATA_DIR = os.path.join(PROJECT_DIR, "data/raw/OxfordPet/images_128x128")

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
local_files_count = len(os.listdir(LOCAL_DATA_DIR))
print(f"Current local files: {local_files_count}/7349")
if local_files_count < 7349:
    print("Caching Oxford-IIIT Pet images to local NVMe disk...")
    t0 = time.time()
    !cp -r "$DRIVE_DATA_DIR"/* "$LOCAL_DATA_DIR"/
    print(f"Caching complete in {time.time() - t0:.1f}s. Local count: {len(os.listdir(LOCAL_DATA_DIR))}")
else:
    print("Oxford-IIIT Pet images already fully cached on local NVMe.")

In [ ]:
# Step 3: Setup Datasets & DataLoaders
import torch
from torch.utils.data import DataLoader
from data.oxford_pet import OxfordPetDataset, collate_oxford

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
MANIFESTS_DIR = os.path.join(PROJECT_DIR, "configs/manifests")

train_dataset = OxfordPetDataset(
    manifest_path=MANIFESTS_DIR,
    images_dir=LOCAL_DATA_DIR,
    split="train"
)
val_dataset = OxfordPetDataset(
    manifest_path=MANIFESTS_DIR,
    images_dir=LOCAL_DATA_DIR,
    split="val"
)
test_dataset = OxfordPetDataset(
    manifest_path=MANIFESTS_DIR,
    images_dir=LOCAL_DATA_DIR,
    split="test"
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_oxford, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_oxford, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_oxford, num_workers=0, pin_memory=True)

print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)} | Test samples: {len(test_dataset)}")

In [ ]:
# Step 4: Define Paths to Warm-Start Weights from Task 2
WARMSTART_PATHS = {
    "classifier": os.path.join(PROJECT_DIR, "checkpoints/task2/best_classifier.pth"),
    "sp": os.path.join(PROJECT_DIR, "checkpoints/task2/best_specialist_salt_and_pepper.pth"),
    "blur": os.path.join(PROJECT_DIR, "checkpoints/task2/best_specialist_gaussian_blur.pth"),
    "occlusion": os.path.join(PROJECT_DIR, "checkpoints/task2/best_specialist_rectangular_occlusion.pth")
}

print("Checking warm-start checkpoints:")
for name, path in WARMSTART_PATHS.items():
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / (1024*1024) if exists else 0
    status = f"[FOUND - {size_mb:.2f} MB]" if exists else "[WARNING: NOT FOUND]"
    print(f"  - {name:<12}: {status} {path}")

In [ ]:
# Step 5: Optuna Hyperparameter Optimization Study (15 Trials)
import optuna
from training.optuna_moe import run_optuna_moe_study

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = run_optuna_moe_study(
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    warmstart_paths=WARMSTART_PATHS,
    n_trials=15,
    epochs_per_trial=4,
    study_name="task3_soft_moe_optimization"
)

best_params = study.best_params
print("\nSelected Optimal Hyperparameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Step 6: Full 2-Phase Training of SoftMoERestorationNetwork
import mlflow
from models.moe import SoftMoERestorationNetwork
from training.trainer_moe import train_soft_moe

mlflow.set_tracking_uri("sqlite:///mlruns.db")
mlflow.set_experiment("Task3_Soft_Mixture_of_Experts")

# Instantiate model and warm-start weights
soft_moe_model = SoftMoERestorationNetwork(
    temperature=best_params.get("temperature", 1.0)
).to(device)

status = soft_moe_model.load_warmstart_weights(
    classifier_path=WARMSTART_PATHS["classifier"],
    sp_path=WARMSTART_PATHS["sp"],
    blur_path=WARMSTART_PATHS["blur"],
    occlusion_path=WARMSTART_PATHS["occlusion"],
    device=str(device)
)
print(f"Warm-start load status: {status}")

with mlflow.start_run(run_name="Soft_MoE_2Phase_Final"):
    mlflow.log_params(best_params)
    results = train_soft_moe(
        model=soft_moe_model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        save_dir=os.path.join(PROJECT_DIR, "checkpoints/task3"),
        warmup_epochs=5,
        joint_epochs=30,
        lr_warmup=5e-4,
        lr_joint=best_params.get("lr_joint", 1e-4),
        temperature=best_params.get("temperature", 1.0),
        lambda_recon=0.8,
        lambda_class=best_params.get("lambda_class", 0.2),
        lambda_balance=best_params.get("lambda_balance", 0.1),
        lambda_entropy=best_params.get("lambda_entropy", 0.01),
        alpha=best_params.get("alpha", 0.90)
    )
    mlflow.log_metric("best_val_psnr", results["best_val_psnr"])
    mlflow.log_metric("best_epoch", results["best_epoch"])

In [ ]:
# Step 7: 4-Way Comparative Benchmark (Universal vs Oracle Hard vs Pred Hard vs Soft MoE)
from models.autoencoders import UniversalAutoencoder
from models.classifiers import CorruptionClassifier
from models.hard_router import HardRoutingRestorationPipeline
from evaluation.benchmark_moe import run_4way_benchmark

# 1. Load Universal Baseline (Task 1)
universal_model = UniversalAutoencoder(base_channels=48, bottleneck_dim=96, dropout_rate=0.2).to(device)
task1_ckpt_path = os.path.join(PROJECT_DIR, "checkpoints/task1_universal_best.pth")
if os.path.exists(task1_ckpt_path):
    ckpt = torch.load(task1_ckpt_path, map_location=device)
    universal_model.load_state_dict(ckpt.get("model_state_dict", ckpt))
    print("Loaded Task 1 Universal Autoencoder.")
else:
    print(f"Warning: Task 1 checkpoint not found at {task1_ckpt_path}")

# 2. Load Task 2 Classifier and Specialists for Hard Router
classifier_model = CorruptionClassifier(base_channels=32, dropout_rate=0.2).to(device)
if os.path.exists(WARMSTART_PATHS["classifier"]):
    ckpt = torch.load(WARMSTART_PATHS["classifier"], map_location=device)
    classifier_model.load_state_dict(ckpt.get("model_state_dict", ckpt))

specialist_models = {}
for c_name, path in [("salt_and_pepper", WARMSTART_PATHS["sp"]), ("gaussian_blur", WARMSTART_PATHS["blur"]), ("rectangular_occlusion", WARMSTART_PATHS["occlusion"])]:
    sp_net = UniversalAutoencoder(base_channels=48, bottleneck_dim=96, dropout_rate=0.2).to(device)
    if os.path.exists(path):
        ckpt = torch.load(path, map_location=device)
        sp_net.load_state_dict(ckpt.get("model_state_dict", ckpt))
    specialist_models[c_name] = sp_net

hard_router = HardRoutingRestorationPipeline(
    classifier=classifier_model,
    specialists=specialist_models,
    device=device
)

# 3. Load Trained Soft MoE Model
soft_moe_ckpt_path = os.path.join(PROJECT_DIR, "checkpoints/task3/best_soft_moe.pth")
if os.path.exists(soft_moe_ckpt_path):
    ckpt = torch.load(soft_moe_ckpt_path, map_location=device)
    soft_moe_model.load_state_dict(ckpt["model_state_dict"])
    print("Loaded trained Soft MoE model.")

benchmark_results = run_4way_benchmark(
    universal_model=universal_model,
    hard_router=hard_router,
    soft_moe=soft_moe_model,
    test_loader=test_loader,
    output_json_path=os.path.join(PROJECT_DIR, "configs/benchmarks/task3_moe_benchmark_results.json"),
    device=str(device)
)

In [ ]:
# Step 8: Gating Network Weight Distribution & Routing Heatmap Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

gating_analysis = benchmark_results.get("gating_analysis", {})
weights_matrix = np.array(gating_analysis.get("mean_weights_matrix", np.eye(4)))
class_names = ["Clean", "Salt & Pepper", "Gaussian Blur", "Occlusion"]
expert_names = ["Clean (Identity)", "S&P Specialist", "Blur Specialist", "Occlusion Specialist"]

plt.figure(figsize=(9, 7))
sns.heatmap(
    weights_matrix,
    annot=True,
    fmt=".3f",
    cmap="Blues",
    xticklabels=expert_names,
    yticklabels=class_names,
    cbar_kws={"label": "Average Expert Routing Weight"}
)
plt.title("Soft MoE Gating Network: Mean Expert Weights by True Corruption", fontsize=14, pad=12)
plt.xlabel("Restoration Expert Branch", fontsize=12)
plt.ylabel("Ground-Truth Corruption Type", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "moe_gating_heatmap.png"), dpi=200)
plt.show()
print("Saved routing heatmap to moe_gating_heatmap.png")

In [ ]:
# Step 9: Self-Contained ONNX Export & Parity Verification
from models.onnx_export import export_to_onnx, verify_onnx_numerical_equivalence

soft_moe_model.eval().cpu()
onnx_save_path = os.path.join(PROJECT_DIR, "models/task3_soft_moe.onnx")

export_to_onnx(
    model=soft_moe_model,
    onnx_output_path=onnx_save_path,
    input_shape=(1, 3, 128, 128),
    opset_version=18,
    input_names=["corrupted_image"],
    output_names=["restored_image", "routing_weights", "logits"]
)

verification = verify_onnx_numerical_equivalence(
    model=soft_moe_model,
    onnx_path=onnx_save_path,
    sample_batch=torch.rand(2, 3, 128, 128),
    atol=1e-3,
    rtol=1e-2
)
print(f"\nSoft MoE ONNX Parity Status: {verification.get('parity_status', verification.get('equivalent'))}")